In [148]:
import sys  
sys.path.insert(1, '../src')

from pathlib import Path
import pandas as pd

In [149]:
PROJECT_ROOT = Path.cwd().parent
REFS_ROOT = Path("/Volumes/EXT DATA/IDMB/")
print(f'Racine du projet : {PROJECT_ROOT}')
print(f'Racine des refs : {REFS_ROOT}')

Racine du projet : /Users/superbg/Documents/IEES/idmybee/data
Racine des refs : /Volumes/EXT DATA/IDMB


In [150]:
terrain = pd.read_csv(REFS_ROOT / 'IDMB_Bombus_terrain.csv')
collect = pd.read_csv(REFS_ROOT / 'IDMB_Bombus_collection.csv')

In [151]:
sorted(terrain.columns)

['capture_locality',
 'dd',
 'device',
 'ia1_id',
 'ia1_top1',
 'ia1_top3',
 'ia2_id',
 'ia2_top1',
 'ia2_top3',
 'ia3_id',
 'ia3_top1',
 'ia3_top3',
 'identifier',
 'identifier_level',
 'inv_id',
 'mm',
 'n_photos',
 'onsite_id',
 'onsite_id_time',
 'photographer',
 'protocol_comment',
 'protocol_time',
 'species',
 'yyyy']

In [152]:
terrain["device"].value_counts()

device
Pixel 7 Pro            19
Huawai mate 10 lite    11
Redmi ai camera        11
Huawei P30             11
LG K40                 10
Samsung S10            10
Huawei P30 lite        10
Samsung s10            10
Fairphone 3+           10
Iphone 12              10
Samsung A52S           10
Huawei P30              1
Name: count, dtype: int64

In [153]:
terrain["device_type"] = "S"

In [154]:
collect["device"].value_counts()

device
Smartphone Samsung s10         748
Canon EOS 70D - 100mm macro    330
Canon EOS 70D - 40mm           324
Canon PowerShot G16             35
Name: count, dtype: int64

In [155]:
collect["device_type"] = "P"
collect.loc[collect["device"] == "Smartphone Samsung s10", "device_type"] = "S"

In [156]:
terrain['species'] = terrain["species"].str.lower()
terrain['onsite_id'] = terrain["species"].str.lower()

def onsite_time_s(row):
    try:
        m, s = row.onsite_id_time.split(':')
        return int(m) * 60 + int(s)
    except:
        return None

def protocol_time_s(row):
    try:
        m, s = row.protocol_time.split(':')
        return int(m) * 60 + int(s)
    except:
        return None

terrain['onsite_id_time_s'] = terrain.apply(onsite_time_s, axis=1)
terrain['protocol_time_s'] = terrain.apply(protocol_time_s, axis=1)

collect['species'] = collect["species"].str.lower()
collect['caste'] = collect["caste"].str.lower()
collect.caste = collect.caste.replace(
    ["mâle", "reine", "femelle", "ouvrière"],
    ["drone", "queen", "worker", "worker"]
)
collect.dd = collect.dd.astype('Int64')
collect.mm = collect.mm.astype('Int64')
collect.yyyy = collect.yyyy.astype('Int64')
collect.identification_year = collect.identification_year.astype('Int64')


In [157]:
terrain["genus"] = "Bombus"
terrain["caste"] = ""
terrain["specimen_state"] = "alive"
terrain["device_type"] = "S"

In [158]:
terrain["source_type"] = "terrain"
collect["source_type"] = "collection"

In [159]:
collect['inv_num'] = 0

In [160]:
terrain_cols = [
    'inv_id',
    'source_type',

    'genus',
    'species',
    'caste',
    'identifier_level',
    'onsite_id',
    'onsite_id_time_s',
    'protocol_time_s',
    'protocol_comment',

    'ia1_id',
    'ia1_top1',
    'ia1_top3',
    'ia2_id',
    'ia2_top1',
    'ia2_top3',
    'ia3_id',
    'ia3_top1',
    'ia3_top3',

    'dd',
    'mm',
    'yyyy',
    'capture_locality',
    'photographer',
    'specimen_state',
    'device',
    'device_type',
    'n_photos',
]
terrain = terrain[terrain_cols]

In [161]:
collect["original_id"] = collect["inv_id"]

In [ ]:
collect_cols = [
    'inv_id',
    'source_type',

    'dd',
    'mm',
    'yyyy',
    'collection_origin',
    'capture_mode',
    'capture_region',
    'capture_locality',

    'collector',

    'order',
    'family',
    'genus',
    'species',
    'caste',
    'identifier',
    'identification_year',

    'photographer',
    'specimen_state',
    'device',
    'device_type',
    'n_photos'
]
collect = collect[collect_cols]
collect = collect.sort_values(['inv_id', 'yyyy', 'mm', 'dd'])

In [166]:
collect.to_csv(PROJECT_ROOT / 'identification' / 'IDMB_Bombus_collect.csv', index=False)
terrain.to_csv(PROJECT_ROOT / 'identification' / 'IDMB_Bombus_terrain.csv', index=False)